# Analyse generated and ranked sequence results

imports and settings

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

from scipy.stats import spearmanr, permutation_test, brunnermunzel, ks_2samp
from roc_boot import plot_prc_with_bands, plot_roc_with_bands
from pathlib import Path
from vsd.proposals import LSTMProposal

from utils import seq2intarr

plt.rcParams['legend.fontsize'] = 8

RESPATH = Path("../results/lstm")
GENFILE = "generated_upo.csv"
RANKFILE = "ranked_variants.csv"
EXPFILE = "../data/UPO for CSIRO 20250827.xlsx"
PTRFILE_RG = "../data/subset_ptm_results_RG_series.csv"
PTRFILE_VAL = "../data/subset_ptm_results_orig_val_10.csv"
AGGREGATE = False
FAST_R = False  # use the fast, but more biased method for computing Spearman R

WTSEQ = (
    "MKTTTLLFLVGALTQTHAFPQQGVPHPLPWSPPGPNDVRAPCPMLNTLANHGFLPHNGKNITQQHTINALYNAL"
    "GIDAELATYLHQEAVTTNPVPNATTFSLNDLSRHDILEHDASLSRQDAFFGDNHDFNQTIFNQTRSYWTSPIID"
    "VKQAALARQARVNTSMATNPNYTMSELGDAFSYGETAAYIIVLGDKEAGLVNRSRVEYLFENERLPVELGWSRA"
    "RENITFDDLSTMLNKIINATGGESEFERELAKRGGVHVGWR"
)
WTARR = np.array(list(WTSEQ))
MUT_SITES = np.array([8, 28, 65, 84, 129, 177, 179, 182, 186, 231]) - 1

GMUTS = [
    "F8G",
    "H65C",
    "L84V_D129N",
    "L84E_G182I",
    "L84Q_L231V",
    "L84E_A186V_L231V",
    "L84M_G182F_L231V",
    "F8G_G182E_L231V",
    "F8I_H65F_D129A_D177M",
    "H65K_F179E_L231K"
]

RMUTS = [
    "H65R_L84T",
    "L28C_H65R_L84S_D129Q_D177S",
    "F8M_L28E_H65S_D129E",
    "F8S_L28V_H65P_D129S_G182D",
    "L84M_A186Y_L231V",
    "F8M_L28V_L84V_F179K_A186Q_L231A",
    "L84Y_G182P",
    "L84V_D129E",
    "L28R_D129K_G182V",
    "F179T_G182R_L231P"
]

## Useful functions

In [ ]:
def seq2mut(seq: str) -> str:
   seqarr = np.array(list(seq))
   inds = np.where(WTARR != seqarr)[0]
   mutseq = [f"{WTSEQ[i]}{i+1}{seq[i]}" for i in inds]
   return "_".join(mutseq)

# Turn mutations into a sequence
def muts2seq(muts: str, wt: str = WTSEQ) -> str:
    if muts == "WT" or pd.isna(muts):
        return wt
    muts = muts.split("_")
    seq = wt
    for m in muts:
        o, ind, n = m[0], int(m[1:-1]) - 1, m[-1]
        if o != seq[ind]:
            raise RuntimeError(f"Substitution {o} @ {ind+1} does not match WT!")
        seq = seq[:ind] + n + seq[ind + 1 :]
    return seq

## Load data, get mutations

In [ ]:
dfg = pd.read_csv(RESPATH / GENFILE)
dfr = pd.read_csv(RESPATH / RANKFILE)

In [ ]:
dfg["Mutation"] = dfg["Variants"].apply(seq2mut)
dfr["Mutation"] = dfr["Variant"].apply(seq2mut)

## Sort Generated data, find ranks of selections

In [ ]:
dfg = dfg.sort_values(by="Vdist_ll", ascending=False)
dfg["Rank"] = np.arange(len(dfg))
dfgm = dfg.set_index("Mutation")

In [ ]:
dfgm.loc[GMUTS]

## Remove training sequences from generated sequences, re-rank

In [ ]:
unique_muts = set(dfgm.index).difference(set(dfr["Mutation"]))
dfgm_unique = dfgm.loc[list(unique_muts)]
dfgm_unique.sort_values(by="Vdist_ll", ascending=False, inplace=True)
dfgm_unique["uRank"] = np.arange(len(dfgm_unique))

In [ ]:
dfgm_unique.loc[GMUTS].drop(columns=["Variants"])

In [ ]:
len(dfgm_unique)

## Extract ranked selections

In [ ]:
ranks = np.array([1, 2, 3, 4, 5, 101, 1000, 10000, 20000, 30000]) - 1
dfr = dfr.sort_values(by=["rank"], ascending=True)
dfr.iloc[ranks]

Getting some discrepencies, lets have a look

In [ ]:
dfr = dfr.set_index("Mutation")
dfr.rename(columns={"rank": "agg_rank", "Variant": "Variants"}, inplace=True)
dfr["rank"] = np.arange(len(dfr)) + 1
dfr_selection = dfr.loc[RMUTS]

## Combine and save

In [ ]:
combined_df = dfgm_unique.loc[GMUTS].drop(columns=["CPE_ll", "Acq"])
combined_df["Strategy"] = "G"
combined_df.rename(columns={"Rank": "original_rank", "uRank": "rank"}, inplace=True)
combined_df["rank"] += 1

In [ ]:
combined_df

In [ ]:
dfr_select = dfr_selection[["Variants", "agg_rank", "rank"]].copy()
dfr_select["Strategy"] = "R"

In [ ]:
dfr_select

In [ ]:
combined_df = pd.concat((combined_df, dfr_select))
combined_df.to_csv(RESPATH / "selection.csv", encoding="utf-8-sig")

In [ ]:
combined_df

# Experimental correlations

Get the experimental data in order

In [ ]:
df_exp = pd.read_excel(Path(EXPFILE))
df_exp = df_exp.rename(columns={"Mutations": "Mutation"})
exp_cols = [
    'Mutation',
    'Phenylacetaldehyde (AUC)',
    'Styrene oxide (AUC)',
    'Protein (ng/uL)',
    'Phenylacetaldehyde (μM)',
    'Styrene oxide (μM)',
    'Styrene products (μM)',
    'PA (pmol/μg)',
    'SO (pmol/μg)',
]
df_exp = df_exp[exp_cols]

# Separate substrates, and drop irrelevent cols

# Aggregate results
if AGGREGATE:
    df_exp = df_exp.groupby(["Mutation"]).mean()

# Compute variants
df_exp.loc[df_exp["Mutation"].isna(), "Mutation"] = "WT"
df_exp["Variants"] = df_exp["Mutation"].apply(muts2seq)

In [ ]:
# Recompute specificity ratio (Styrene Oxide Index)
def aspec(A, B, eps=1e-5):
    A = np.maximum(0, A)
    B = np.maximum(0, B)
    ratio = 2 * A / (A + B + eps) - 1
    return ratio

df_exp["aspec(SO, PA)"] = aspec(df_exp["Styrene oxide (μM)"], df_exp["Phenylacetaldehyde (μM)"])
exp_cols += ["aspec(SO, PA)"]

### Load the pretrained models

In [ ]:
df_ptm_rg = pd.read_csv(PTRFILE_RG)
df_ptm_val = pd.read_csv(PTRFILE_VAL)

df_ptm_rg.loc[df_ptm_rg["Mutations"].isna(), "Mutations"] = "WT"
df_ptm_val.loc[df_ptm_val["Mutations"].isna(), "Mutations"] = "WT"

In [ ]:
def format_ptm_data(df):
    df = df.pivot(
        index="Mutations",
        columns="substrate",
        values=[
            "selfprot_finetuned_kcat",
            "selfprot_finetuned_km",
            "netsolp_solubility",
            "netsolp_usability",
            "clipzyme_score",
            "ESP_score",
            "boltz_unconstrained_affinity",
            "boltz_unconstrained_cls",
            "boltz_constrained_affinity",
            "boltz_constrained_cls"
    ])
    df.columns = ["_".join(map(str, c)).strip("_") for c in df.columns.to_flat_index()]
    df.rename(columns={
        "netsolp_solubility_styrene": "netsolp_solubility",
        "netsolp_usability_styrene": "netsolp_usability",
    }, inplace=True)
    df = df.drop(columns=[
        "netsolp_solubility_styrene_oxide",
        "netsolp_solubility_propylbenzene",
        "netsolp_usability_styrene_oxide",
        "netsolp_usability_propylbenzene"
    ])
    df = df.reset_index().rename(columns={"Mutations": "Mutation"})
    return df

df_ptm_rg = format_ptm_data(df_ptm_rg)
df_ptm_val = format_ptm_data(df_ptm_val)

# Merge into experimental results
df_exp = df_exp.merge(df_ptm_rg, on="Mutation", how="outer")

### Get likelihoods from prior, finetuned and variational LSTMs

Load models

In [ ]:
DEVICE = "cpu"
GEN_PARAMS = dict(
    d_features=len(WTSEQ),
    k_categories=20,
    num_layers=2,
    hidden_size=128,
    clip_gradients=1.0
)

# Prior
sd = torch.load(
    f=RESPATH / "prior_LSTMProposal.pt",
    map_location=torch.device(DEVICE),
    weights_only=True,
)
prior = LSTMProposal(**GEN_PARAMS)
prior.load_state_dict(sd)
prior = prior.to(DEVICE)

# Finetuned
sd = torch.load(
    f=RESPATH / "finetuned_LSTMProposal.pt",
    map_location=torch.device(DEVICE),
    weights_only=True,
)
ftmod = LSTMProposal(**GEN_PARAMS)
ftmod.load_state_dict(sd)
ftmod = ftmod.to(DEVICE)

# Posterior
sd = torch.load(
    f=RESPATH / "vard_LSTMProposal.pt",
    map_location=torch.device(DEVICE),
    weights_only=True,
)
vdist = LSTMProposal(**GEN_PARAMS)
vdist.load_state_dict(sd)
vdist = vdist.to(DEVICE)

Query models

In [ ]:
X = seq2intarr(df_exp["Variants"].values)
df_exp["VSD_LSTM"] = vdist.log_prob(X.to(DEVICE)).detach().numpy()
df_exp["FT_LSTM"] = ftmod.log_prob(X.to(DEVICE)).detach().numpy()
df_exp["Prior_LSTM"] = prior.log_prob(X.to(DEVICE)).detach().numpy()

Compute correlations

In [ ]:
corr = pd.DataFrame()
model_cols = ["VSD_LSTM", "FT_LSTM", "Prior_LSTM"] + [c for c in df_ptm_rg.columns if c != "Mutation"]
for i in model_cols:
    for j in exp_cols:
        if j == "Mutation":
            continue
        x = df_exp[[i, j]]
        if FAST_R:
            r, p = spearmanr(x)
        else:
            stat = lambda y: spearmanr(y, x[j]).statistic
            res = permutation_test((x[i],), stat, permutation_type='pairings')
            r, p = res.statistic, res.pvalue
        corr.loc[j, f"{i}"] = f"{r:.3f} ({p:.3f})"
        print(f"{i}, {j}: r = {corr.loc[j, f"{i}"]}")
corr.to_csv(RESPATH / "res_corr.csv", encoding="utf-8-sig")

In [ ]:
all_res = df_exp.merge(combined_df[["Strategy", "rank"]], on="Mutation", how="outer")
all_res.to_csv(RESPATH / "all_results.csv", encoding="utf-8-sig")
all_res = all_res.set_index("Mutation")

In [ ]:
corr

## Compare against original validation

In [ ]:
VALIDATIONPATH = Path("../data/AbrUPO validation.xlsx")
VALIDATION_SHEET = {
    "name": "Sheet1",
    "nrows": 21
}
df_val = pd.read_excel(VALIDATIONPATH, sheet_name=VALIDATION_SHEET["name"],
                       nrows=VALIDATION_SHEET["nrows"])
df_val.rename(columns={"Mutations": "Mutation"}, inplace=True)
df_val["Variants"] = df_val["Mutation"].map(lambda s: muts2seq(s, wt=WTSEQ))
df_val = df_val.drop(columns=[
    "Variant number",
    "PBallEF",
    "PB3EF",
    "PB1EF",
    "PB2EF",
    "PB3(B/D)",
    "PB1(B/D)",
    "PB2(B/D)",
    "SD1 Read count",
    "SD1 percentage"
]).dropna()

# Merge ptms into experimental results
df_val = df_val.merge(df_ptm_val, on="Mutation", how="outer")

In [ ]:
X = seq2intarr(df_val["Variants"].values)
df_val["VSD_LSTM"] = vdist.log_prob(X.to(DEVICE)).detach().numpy()
df_val["Prior_LSTM"] = prior.log_prob(X.to(DEVICE)).detach().numpy()
df_val["FT_LSTM"] = ftmod.log_prob(X.to(DEVICE)).detach().numpy()

df_val.drop(columns="Variants").to_csv(RESPATH / "val_results.csv", encoding="utf-8-sig")

In [ ]:
quants = [
    'Protein concentration (ng/μL)',
    'Phenylacetalehyde (AUC)',
    'Styrene oxide (AUC)',
    'Phenylacetaldehyde (μM)',
    'Styrene oxide (μM)',
    'All styrene products (μM)'
]
corr_val = pd.DataFrame()
for i in model_cols:
    for j in quants:
        x = df_val[[i, j]]
        if FAST_R:
            r, p = spearmanr(x)
        else:
            stat = lambda y: spearmanr(y, x[j]).statistic
            res = permutation_test((x[i],), stat, permutation_type='pairings')
            r, p = res.statistic, res.pvalue
        corr_val.loc[j, f"{i}"] = f"{r:.3f} ({p:.3f})"
        print(f"{i}, {j}: r = {corr_val.loc[j, f"{i}"]}")
corr_val.to_csv(RESPATH / "val_corr.csv", encoding="utf-8-sig")
corr_val

## Selection performance measures

In [ ]:
interesting_models = [
    "VSD_LSTM",
    # "Prior_LSTM",
    # "FT_LSTM",
    # "netsolp_solubility",
    # "netsolp_usability",
    # "boltz_unconstrained_cls_propylbenzene",
    # "boltz_unconstrained_cls_styrene_oxide",
    # "boltz_constrained_affinity_styrene",
    # "ESP_score_propylbenzene",
]

def norm_preds(preds):
    preds = preds - preds.min()
    preds /= preds.max()
    return preds

y_pred = {name: norm_preds(all_res[name]) for name in interesting_models}

### Compare Generative Vs Rank agg selections

In [ ]:
mean_cols = [
    "Styrene oxide (AUC)",
    "Phenylacetaldehyde (AUC)",
    "Styrene oxide (μM)",
    "Phenylacetaldehyde (μM)",
    "PA (pmol/μg)",
    "SO (pmol/μg)",
    "aspec(SO, PA)",
    "Strategy"
]

top5res = all_res.loc[all_res["rank"]<=5, mean_cols]
top5mean = top5res.groupby("Strategy").mean()
for col in mean_cols:
    if col == "Strategy":
        continue
    x = top5res.loc[top5res["Strategy"] == "G", col]
    y = top5res.loc[top5res["Strategy"] == "R", col]
    res = brunnermunzel(x, y, alternative="greater")
    top5mean.loc["Brunner Munzel p-value", col] = res.pvalue
    res = ks_2samp(x, y, alternative="less")
    top5mean.loc["Kolmogorov-Smirnov p-value", col] = res.pvalue

top5mean.to_csv(RESPATH / "top5mean.csv", float_format='%.4g', encoding="utf-8-sig")

allmean = all_res[mean_cols].groupby("Strategy").mean()
for col in mean_cols:
    if col == "Strategy":
        continue
    x = all_res.loc[all_res["Strategy"] == "G", col]
    y = all_res.loc[all_res["Strategy"] == "R", col]
    res = brunnermunzel(x, y, alternative="greater")
    allmean.loc["Brunner Munzel p-value", col] = res.pvalue
    res = ks_2samp(x, y, alternative="less")
    allmean.loc["Kolmogorov-Smirnov p-value", col] = res.pvalue

allmean.to_csv(RESPATH / "allmean.csv", float_format='%.4g', encoding="utf-8-sig")

In [ ]:
top5mean

In [ ]:
allmean

## Model retrospective analysis

### Label generation

In [ ]:
wt = lambda x: x.loc["WT"].max()
wt_total_act = all_res.loc["WT", [
    "Styrene oxide (μM)",
    "Phenylacetaldehyde (μM)"
]].max(axis=0).sum()
wt_specif_act = all_res.loc["WT", [
    "PA (pmol/μg)",
    "SO (pmol/μg)"
]].max(axis=0).sum()

labels = {
    "All AUC > 1": lambda df: (df[[
        "Styrene oxide (AUC)",
        "Phenylacetaldehyde (AUC)"
    ]] > 1).all(axis=1),
    "Styrene Activity": lambda df: df["Styrene oxide (μM)"] > wt(df["Styrene oxide (μM)"]),
    "Styrene Specfic Activity": lambda df: df["SO (pmol/μg)"] > wt(df["SO (pmol/μg)"]),
    "Total Activity": lambda df: df[[
        "Styrene oxide (μM)",
        "Phenylacetaldehyde (μM)"
    ]].sum(axis=1) > wt_total_act,
    "Total Specific Activity": lambda df: df[[
        "PA (pmol/μg)",
        "SO (pmol/μg)"
    ]].sum(axis=1) > wt_specif_act,
    "SOI(SO, PA) > WT": lambda df: df["aspec(SO, PA)"] > wt(df["aspec(SO, PA)"]),
    "SOI(SO, PA) > .5": lambda df: df["aspec(SO, PA)"] > .5,
    "Activity Styrene oxide (μM) > WT": lambda df: df["Styrene oxide (μM)"] > wt(df["Styrene oxide (μM)"]),
}

In [ ]:
print("SOI(SO,PA)-WT: ", wt(all_res["aspec(SO, PA)"]))
print("Styrene oxide (μM)-WT: ", wt(all_res["Styrene oxide (μM)"]))

### Compute metrics for each label

In [ ]:
# Compute metrics
df_prc = pd.DataFrame()
df_roc = pd.DataFrame()
for name, lf in labels.items():
    z = lf(all_res)
    print(f"{name}, +ve labels {z.sum()}")
    pz = z.mean()
    col_name = f"{name}, p(z) = {pz:.3f}"

    fig, ax = plt.subplots(figsize=(7, 7), dpi=150)
    _, prc_info = plot_prc_with_bands(
        y_true=z,
        preds=y_pred,
        ci=0.95,
        n_boot=1000,
        ax=ax,
    )
    plt.title(f"{name}, PRC / AP, positive label proportion: {pz:.3f}")
    plt.show()
    for i, d in prc_info.items():
        df_prc.loc[i, col_name] = f"{d["ap"]:.3f} ({d["ci_low"]:.3f}, {d["ci_high"]:.3f})"

    fig, ax = plt.subplots(figsize=(7, 7), dpi=150)
    _, roc_info = plot_roc_with_bands(
        y_true=z,
        preds=y_pred,
        ci=0.95,
        n_boot=1000,
        ax=ax,
    )
    plt.title(f"{name}, ROC / AUC, +ve label proportion: {pz:.3f}")
    plt.show()
    for i, d in roc_info.items():
        df_roc.loc[i, col_name] = f"{d["auc"]:.3f} ({d["ci_low"]:.3f}, {d["ci_high"]:.3f})"

In [ ]:
df_prc.to_csv(RESPATH / "PRC_results.csv", encoding="utf-8-sig")
df_prc

In [ ]:
df_roc.to_csv(RESPATH / "ROC_results.csv", encoding="utf-8-sig")
df_roc